# Ingest — Verification Notebook
Mirrors `ingest.py` step-by-step so you can inspect outputs at each stage.

In [14]:
print("hello")

hello


In [15]:
!pip install pandas

In [16]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR  = Path('data')
CSV_PATH  = DATA_DIR / 'kaggle_arxiv.csv'
JSON_PATH = DATA_DIR / 'papers_raw.json'
DB_PATH   = DATA_DIR / 'arxiv.db'

SUPPORTED_CATEGORIES = {'cs.AI', 'cs.LG', 'cs.CL', 'stat.ML', 'cs.CV'}

## 1. Load raw CSV

In [17]:
df = pd.read_csv(CSV_PATH)
print('Raw shape:', df.shape)
df.head()

Raw shape: (100000, 13)


,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,update_date,abstract,submitted,version_count
0,2407.00993,Weikai Xu,"Shihan Deng, Weikai Xu, Hongda Sun, Wei Liu, T...",Mobile-Bench: An Evaluation Benchmark for LLM-...,NaN,NaN,NaN,NaN,cs.AI cs.CL,2024-07-02,With the remarkable advancements of large lang...,"Mon, 1 Jul 2024 06:10:01 GMT",1
1,1304.114,Linda C. van der Gaag,Linda C. van der Gaag,Computing Probability Intervals Under Independ...,Appears in Proceedings of the Sixth Conference...,NaN,NaN,UAI-P-1990-PG-491-497,cs.AI,2013-04-05,Many AI researchers argue that probability the...,"Wed, 27 Mar 2013 14:00:05 GMT",1
2,2205.09971,Yacine Izza,"Yacine Izza, Alexey Ignatiev and Joao Marques-...",On Tackling Explanation Redundancy in Decision...,NaN,J. Artif. Intell. Res. Vol. 75 (2022),10.1613/jair.1.13575,NaN,cs.AI cs.LG,2022-10-04,Decision trees (DTs) epitomize the ideal of in...,"Fri, 20 May 2022 05:33:38 GMT",2
3,1211.2719,Norbert B\'atfai Ph.D.,N. B\'atfai,Quantum Consciousness Soccer Simulator,"9 pages, grammatically improved",NaN,NaN,NaN,cs.AI cs.MA,2012-11-14,In cognitive sciences it is not uncommon to us...,"Mon, 12 Nov 2012 18:10:05 GMT",2
4,2502.11574,Birger Moell,"Johan Boye, Birger Moell",Large Language Models and Mathematical Reasoni...,NaN,NaN,NaN,NaN,cs.AI,2025-02-24,This paper investigates the mathematical reaso...,"Mon, 17 Feb 2025 09:07:32 GMT",2


## 2. Filter supported categories

In [18]:
def has_supported_category(cat_string):
    if pd.isna(cat_string):
        return False
    return bool(set(cat_string.split()) & SUPPORTED_CATEGORIES)

mask = df['categories'].apply(has_supported_category)
df = df[mask].reset_index(drop=True)
print('Filtered shape:', df.shape)
df['categories'].value_counts().head(20)

Filtered shape: (100000, 13)


categories
cs.CV                            12025
cs.AI                             9593
cs.CL                             9544
stat.ML cs.LG                     8865
cs.LG                             4788
cs.CL cs.AI                       3733
cs.LG cs.AI                       2800
cs.LG stat.ML                     2538
cs.CV cs.AI                       1845
cs.AI cs.LG                       1630
stat.ML                           1581
cs.CL cs.LG                       1551
cs.CV cs.LG                       1396
cs.CL cs.AI cs.LG                 1252
cs.AI cs.CL                       1091
stat.ML cs.LG stat.ME              704
stat.ML cs.AI cs.LG                670
stat.ML cs.LG math.ST stat.TH      634
cs.LG cs.AI stat.ML                628
cs.CV cs.AI cs.LG                  583
Name: count, dtype: int64

## 3. Build raw_papers table

In [19]:
raw_papers = pd.DataFrame({
    'arxiv_id':         df['id'],
    'title':            df['title'],
    'abstract':         df['abstract'],
    'authors':          df['authors'],
    'categories':       df['categories'],
    'primary_category': df['categories'].str.split().str[0],
    'submitted_date':   pd.to_datetime(df['submitted'],   errors='coerce'),
    'update_date':      pd.to_datetime(df['update_date'], errors='coerce'),
    'journal_ref':      df['journal-ref'],
    'doi':              df['doi'],
    'comment':          df['comments'],
})

print(raw_papers.shape)
print(raw_papers.dtypes)
raw_papers.head()

(100000, 11)
arxiv_id                       str
title                          str
abstract                       str
authors                        str
categories                     str
primary_category            object
submitted_date      datetime64[us]
update_date         datetime64[us]
journal_ref                    str
doi                            str
comment                        str
dtype: object


,arxiv_id,title,abstract,authors,categories,primary_category,submitted_date,update_date,journal_ref,doi,comment
0,2407.00993,Mobile-Bench: An Evaluation Benchmark for LLM-...,With the remarkable advancements of large lang...,"Shihan Deng, Weikai Xu, Hongda Sun, Wei Liu, T...",cs.AI cs.CL,cs.AI,2024-07-01 06:10:01,2024-07-02,NaN,NaN,NaN
1,1304.114,Computing Probability Intervals Under Independ...,Many AI researchers argue that probability the...,Linda C. van der Gaag,cs.AI,cs.AI,2013-03-27 14:00:05,2013-04-05,NaN,NaN,Appears in Proceedings of the Sixth Conference...
2,2205.09971,On Tackling Explanation Redundancy in Decision...,Decision trees (DTs) epitomize the ideal of in...,"Yacine Izza, Alexey Ignatiev and Joao Marques-...",cs.AI cs.LG,cs.AI,2022-05-20 05:33:38,2022-10-04,J. Artif. Intell. Res. Vol. 75 (2022),10.1613/jair.1.13575,NaN
3,1211.2719,Quantum Consciousness Soccer Simulator,In cognitive sciences it is not uncommon to us...,N. B\'atfai,cs.AI cs.MA,cs.AI,2012-11-12 18:10:05,2012-11-14,NaN,NaN,"9 pages, grammatically improved"
4,2502.11574,Large Language Models and Mathematical Reasoni...,This paper investigates the mathematical reaso...,"Johan Boye, Birger Moell",cs.AI,cs.AI,2025-02-17 09:07:32,2025-02-24,NaN,NaN,NaN


## 4. Null / quality check

In [20]:
raw_papers.isnull().sum()

arxiv_id                0
title                   0
abstract                0
authors                 0
categories              0
primary_category        0
submitted_date          0
update_date             0
journal_ref         90895
doi                 89931
comment             43158
dtype: int64

## 5. Export → papers_raw.json

In [21]:
records = raw_papers.copy()
records['submitted_date'] = records['submitted_date'].dt.strftime('%Y-%m-%d').where(records['submitted_date'].notna(), None)
records['update_date']    = records['update_date'].dt.strftime('%Y-%m-%d').where(records['update_date'].notna(), None)
records.to_json(JSON_PATH, orient='records', indent=2, force_ascii=False)
print(f'Saved {len(records):,} records → {JSON_PATH}')

Saved 100,000 records → data/papers_raw.json


## 6. Export → arxiv.db (table: raw_papers)

In [22]:
db_df = raw_papers.copy()
db_df['submitted_date'] = db_df['submitted_date'].dt.strftime('%Y-%m-%d').where(db_df['submitted_date'].notna(), None)
db_df['update_date']    = db_df['update_date'].dt.strftime('%Y-%m-%d').where(db_df['update_date'].notna(), None)

with sqlite3.connect(DB_PATH) as conn:
    db_df.to_sql('raw_papers', conn, if_exists='replace', index=False)
    count = conn.execute('SELECT COUNT(*) FROM raw_papers').fetchone()[0]

print(f'Saved {count:,} rows → table raw_papers in {DB_PATH}')

Saved 100,000 rows → table raw_papers in data/arxiv.db


## 7. Verify DB — quick query

In [23]:
with sqlite3.connect(DB_PATH) as conn:
    verify = pd.read_sql('SELECT * FROM raw_papers LIMIT 5', conn)
verify

,arxiv_id,title,abstract,authors,categories,primary_category,submitted_date,update_date,journal_ref,doi,comment
0,2407.00993,Mobile-Bench: An Evaluation Benchmark for LLM-...,With the remarkable advancements of large lang...,"Shihan Deng, Weikai Xu, Hongda Sun, Wei Liu, T...",cs.AI cs.CL,cs.AI,2024-07-01,2024-07-02,NaN,NaN,NaN
1,1304.114,Computing Probability Intervals Under Independ...,Many AI researchers argue that probability the...,Linda C. van der Gaag,cs.AI,cs.AI,2013-03-27,2013-04-05,NaN,NaN,Appears in Proceedings of the Sixth Conference...
2,2205.09971,On Tackling Explanation Redundancy in Decision...,Decision trees (DTs) epitomize the ideal of in...,"Yacine Izza, Alexey Ignatiev and Joao Marques-...",cs.AI cs.LG,cs.AI,2022-05-20,2022-10-04,J. Artif. Intell. Res. Vol. 75 (2022),10.1613/jair.1.13575,NaN
3,1211.2719,Quantum Consciousness Soccer Simulator,In cognitive sciences it is not uncommon to us...,N. B\'atfai,cs.AI cs.MA,cs.AI,2012-11-12,2012-11-14,NaN,NaN,"9 pages, grammatically improved"
4,2502.11574,Large Language Models and Mathematical Reasoni...,This paper investigates the mathematical reaso...,"Johan Boye, Birger Moell",cs.AI,cs.AI,2025-02-17,2025-02-24,NaN,NaN,NaN


---
## 8. Build `papers` table

In [24]:

def get_first_author(author_string):
    """Return the first author name from a comma-separated author string."""
    if pd.isna(author_string):
        return None
    return author_string.split(',')[0].strip()

def count_authors(author_string):
    """Count number of authors in a comma-separated author string."""
    if pd.isna(author_string):
        return 0
    return len([a for a in author_string.split(',') if a.strip()])

def get_subject_area(primary_category):
    """Map primary category prefix to a broad subject area."""
    if pd.isna(primary_category):
        return None
    prefix = primary_category.split('.')[0].lower()
    mapping = {
        'cs':   'Computer Science',
        'stat': 'Statistics',
        'math': 'Mathematics',
        'eess': 'Electrical Engineering',
        'econ': 'Economics',
        'q-bio': 'Quantitative Biology',
        'q-fin': 'Quantitative Finance',
        'physics': 'Physics',
    }
    return mapping.get(prefix, prefix.upper())

def get_pub_status(row):
    """'Published' if DOI or journal_ref present, otherwise 'Preprint'."""
    has_doi     = pd.notna(row['doi'])        and str(row['doi']).strip()        != ''
    has_journal = pd.notna(row['journal_ref']) and str(row['journal_ref']).strip() != ''
    return 'Published' if (has_doi or has_journal) else 'Preprint'

papers = pd.DataFrame({
    'arxiv_id':           raw_papers['arxiv_id'],
    'title':              raw_papers['title'],
    'abstract':           raw_papers['abstract'],
    'authors':            raw_papers['authors'],
    'primary_category':   raw_papers['primary_category'],
    'submitted':          raw_papers['submitted_date'],
    'abstract_word_count': raw_papers['abstract'].fillna('').apply(lambda x: len(x.split())),
    'author_count':       raw_papers['authors'].apply(count_authors),
    'first_author':       raw_papers['authors'].apply(get_first_author),
    'submitted_year':     raw_papers['submitted_date'].dt.year.astype('Int64'),
    'subject_area':       raw_papers['primary_category'].apply(get_subject_area),
    'pub_status':         raw_papers[['doi', 'journal_ref']].apply(get_pub_status, axis=1),
})

print(papers.shape)
print(papers.dtypes)
papers.head()


(100000, 12)
arxiv_id                          str
title                             str
abstract                          str
authors                           str
primary_category               object
submitted              datetime64[us]
abstract_word_count             int64
author_count                    int64
first_author                      str
submitted_year                  Int64
subject_area                      str
pub_status                        str
dtype: object


,arxiv_id,title,abstract,authors,primary_category,submitted,abstract_word_count,author_count,first_author,submitted_year,subject_area,pub_status
0,2407.00993,Mobile-Bench: An Evaluation Benchmark for LLM-...,With the remarkable advancements of large lang...,"Shihan Deng, Weikai Xu, Hongda Sun, Wei Liu, T...",cs.AI,2024-07-01 06:10:01,205,11,Shihan Deng,2024,Computer Science,Preprint
1,1304.114,Computing Probability Intervals Under Independ...,Many AI researchers argue that probability the...,Linda C. van der Gaag,cs.AI,2013-03-27 14:00:05,89,1,Linda C. van der Gaag,2013,Computer Science,Preprint
2,2205.09971,On Tackling Explanation Redundancy in Decision...,Decision trees (DTs) epitomize the ideal of in...,"Yacine Izza, Alexey Ignatiev and Joao Marques-...",cs.AI,2022-05-20 05:33:38,256,2,Yacine Izza,2022,Computer Science,Published
3,1211.2719,Quantum Consciousness Soccer Simulator,In cognitive sciences it is not uncommon to us...,N. B\'atfai,cs.AI,2012-11-12 18:10:05,94,1,N. B\'atfai,2012,Computer Science,Preprint
4,2502.11574,Large Language Models and Mathematical Reasoni...,This paper investigates the mathematical reaso...,"Johan Boye, Birger Moell",cs.AI,2025-02-17 09:07:32,163,2,Johan Boye,2025,Computer Science,Preprint


## 9. Quality check — papers table

In [25]:

print("Null counts:")
print(papers.isnull().sum())
print("\npub_status distribution:")
print(papers['pub_status'].value_counts())
print("\nsubject_area distribution:")
print(papers['subject_area'].value_counts())
print("\nsubmitted_year range:", papers['submitted_year'].min(), "→", papers['submitted_year'].max())
print("\nabstract_word_count stats:")
print(papers['abstract_word_count'].describe())


Null counts:
arxiv_id               0
title                  0
abstract               0
authors                0
primary_category       0
submitted              0
abstract_word_count    0
author_count           0
first_author           0
submitted_year         0
subject_area           0
pub_status             0
dtype: int64

pub_status distribution:
pub_status
Preprint     85809
Published    14191
Name: count, dtype: int64

subject_area distribution:
subject_area
Computer Science    79999
Statistics          20001
Name: count, dtype: int64

submitted_year range: 1993 → 2026

abstract_word_count stats:
count    100000.000000
mean        166.855020
std          47.154602
min           3.000000
25%         135.000000
50%         165.000000
75%         198.000000
max         558.000000
Name: abstract_word_count, dtype: float64


## 10. Export → arxiv.db (table: papers)

In [26]:

db_papers = papers.copy()
db_papers['submitted'] = db_papers['submitted'].dt.strftime('%Y-%m-%d').where(
    db_papers['submitted'].notna(), None
)
db_papers['submitted_year'] = db_papers['submitted_year'].astype(object).where(
    db_papers['submitted_year'].notna(), None
)

with sqlite3.connect(DB_PATH) as conn:
    db_papers.to_sql('papers', conn, if_exists='replace', index=False)
    count = conn.execute('SELECT COUNT(*) FROM papers').fetchone()[0]

print(f'Saved {count:,} rows → table papers in {DB_PATH}')


Saved 100,000 rows → table papers in data/arxiv.db


## 11. Verify — papers table

In [27]:

with sqlite3.connect(DB_PATH) as conn:
    verify_papers = pd.read_sql('SELECT * FROM papers LIMIT 5', conn)
verify_papers


,arxiv_id,title,abstract,authors,primary_category,submitted,abstract_word_count,author_count,first_author,submitted_year,subject_area,pub_status
0,2407.00993,Mobile-Bench: An Evaluation Benchmark for LLM-...,With the remarkable advancements of large lang...,"Shihan Deng, Weikai Xu, Hongda Sun, Wei Liu, T...",cs.AI,2024-07-01,205,11,Shihan Deng,2024,Computer Science,Preprint
1,1304.114,Computing Probability Intervals Under Independ...,Many AI researchers argue that probability the...,Linda C. van der Gaag,cs.AI,2013-03-27,89,1,Linda C. van der Gaag,2013,Computer Science,Preprint
2,2205.09971,On Tackling Explanation Redundancy in Decision...,Decision trees (DTs) epitomize the ideal of in...,"Yacine Izza, Alexey Ignatiev and Joao Marques-...",cs.AI,2022-05-20,256,2,Yacine Izza,2022,Computer Science,Published
3,1211.2719,Quantum Consciousness Soccer Simulator,In cognitive sciences it is not uncommon to us...,N. B\'atfai,cs.AI,2012-11-12,94,1,N. B\'atfai,2012,Computer Science,Preprint
4,2502.11574,Large Language Models and Mathematical Reasoni...,This paper investigates the mathematical reaso...,"Johan Boye, Birger Moell",cs.AI,2025-02-17,163,2,Johan Boye,2025,Computer Science,Preprint


---
## 12. Build `category_stats` table

In [28]:
category_stats = (
    papers.groupby('primary_category', as_index=False)
    .agg(
        total_papers=('arxiv_id', 'count'),
        published_count=('pub_status', lambda x: (x == 'Published').sum()),
    )
    .rename(columns={'primary_category': 'category'})
)
category_stats['published_rate_pct'] = (
    (category_stats['published_count'] / category_stats['total_papers']) * 100
).round(2)

print(category_stats.shape)
category_stats


(5, 4)


,category,total_papers,published_count,published_rate_pct
0,cs.AI,19999,3295,16.48
1,cs.CL,20000,2293,11.46
2,cs.CV,20001,2727,13.63
3,cs.LG,19999,2652,13.26
4,stat.ML,20001,3224,16.12


## 13. Export → arxiv.db (table: category_stats)

In [29]:
with sqlite3.connect(DB_PATH) as conn:
    category_stats.to_sql('category_stats', conn, if_exists='replace', index=False)
    count = conn.execute('SELECT COUNT(*) FROM category_stats').fetchone()[0]

print(f'Saved {count:,} rows → table category_stats in {DB_PATH}')

with sqlite3.connect(DB_PATH) as conn:
    pd.read_sql('SELECT * FROM category_stats ORDER BY total_papers DESC', conn)


Saved 5 rows → table category_stats in data/arxiv.db


---
## 14. Build `yearly_trends` table

In [30]:
yearly_trends = (
    papers.dropna(subset=['submitted_year'])
    .groupby(['submitted_year', 'primary_category'], as_index=False)
    .agg(paper_count=('arxiv_id', 'count'))
    .rename(columns={'submitted_year': 'year', 'primary_category': 'category'})
    .sort_values(['year', 'category'])
    .reset_index(drop=True)
)
yearly_trends['year'] = yearly_trends['year'].astype(int)

print(yearly_trends.shape)
yearly_trends.head(15)


(137, 3)


,year,category,paper_count
0,1993,cs.AI,5
1,1994,cs.AI,5
2,1995,cs.AI,12
3,1996,cs.AI,19
4,1997,cs.AI,13
5,1998,cs.AI,10
6,1998,cs.CL,9
7,1999,cs.AI,7
8,1999,cs.CL,16
9,1999,cs.LG,1


## 15. Export → arxiv.db (table: yearly_trends)

In [31]:
with sqlite3.connect(DB_PATH) as conn:
    yearly_trends.to_sql('yearly_trends', conn, if_exists='replace', index=False)
    count = conn.execute('SELECT COUNT(*) FROM yearly_trends').fetchone()[0]

print(f'Saved {count:,} rows → table yearly_trends in {DB_PATH}')

with sqlite3.connect(DB_PATH) as conn:
    pd.read_sql('SELECT * FROM yearly_trends ORDER BY year, category LIMIT 15', conn)


Saved 137 rows → table yearly_trends in data/arxiv.db


---
## 16. Build `publication_status` table

In [32]:
publication_status = (
    papers.groupby(['pub_status', 'primary_category'], as_index=False)
    .agg(paper_count=('arxiv_id', 'count'))
    .rename(columns={'primary_category': 'category'})
    .sort_values(['category', 'pub_status'])
    .reset_index(drop=True)
)

print(publication_status.shape)
publication_status


(10, 3)


,pub_status,category,paper_count
0,Preprint,cs.AI,16704
1,Published,cs.AI,3295
2,Preprint,cs.CL,17707
3,Published,cs.CL,2293
4,Preprint,cs.CV,17274
5,Published,cs.CV,2727
6,Preprint,cs.LG,17347
7,Published,cs.LG,2652
8,Preprint,stat.ML,16777
9,Published,stat.ML,3224


## 17. Export → arxiv.db (table: publication_status)

In [33]:
with sqlite3.connect(DB_PATH) as conn:
    publication_status.to_sql('publication_status', conn, if_exists='replace', index=False)
    count = conn.execute('SELECT COUNT(*) FROM publication_status').fetchone()[0]

print(f'Saved {count:,} rows → table publication_status in {DB_PATH}')

with sqlite3.connect(DB_PATH) as conn:
    pd.read_sql('SELECT * FROM publication_status ORDER BY category, pub_status', conn)


Saved 10 rows → table publication_status in data/arxiv.db


---
## 18. Build `author_stats` table

In [34]:
# Drop rows with no first_author
pa = papers.dropna(subset=['first_author'])

# Base aggregation: paper_count, first_year, last_year
base = (
    pa.groupby('first_author', as_index=False)
    .agg(
        paper_count=('arxiv_id', 'count'),
        first_year=('submitted_year', 'min'),
        last_year=('submitted_year', 'max'),
    )
)

# top_category: category with most appearances per author
top_cat = (
    pa.groupby(['first_author', 'primary_category'])
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
    .drop_duplicates(subset='first_author')   # keep highest-count category
    [['first_author', 'primary_category']]
    .rename(columns={'primary_category': 'top_category'})
)

author_stats = (
    base.merge(top_cat, on='first_author', how='left')
    .rename(columns={'first_author': 'author'})
    .sort_values('paper_count', ascending=False)
    .reset_index(drop=True)
)

# clean up year types
author_stats['first_year'] = author_stats['first_year'].astype('Int64')
author_stats['last_year']  = author_stats['last_year'].astype('Int64')

print(author_stats.shape)
author_stats.head(10)


(73254, 5)


,author,paper_count,first_year,last_year,top_category
0,Xiang Li,38,2017,2026,stat.ML
1,Yu Zhang,28,2011,2026,cs.AI
2,Xi Chen,26,2010,2026,stat.ML
3,Yu Wang,24,2019,2026,cs.CL
4,Benyamin Ghojogh,24,2019,2023,stat.ML
5,Wei Chen,23,2014,2026,cs.CV
6,Nathan Kallus,22,2015,2025,stat.ML
7,Yang Liu,22,2013,2026,cs.CV
8,Hao Chen,22,2017,2026,cs.LG
9,Hao Wang,21,2015,2026,cs.CV


## 19. Export → arxiv.db (table: author_stats)

In [35]:
db_author = author_stats.copy()
db_author['first_year'] = db_author['first_year'].astype(object).where(db_author['first_year'].notna(), None)
db_author['last_year']  = db_author['last_year'].astype(object).where(db_author['last_year'].notna(), None)

with sqlite3.connect(DB_PATH) as conn:
    db_author.to_sql('author_stats', conn, if_exists='replace', index=False)
    count = conn.execute('SELECT COUNT(*) FROM author_stats').fetchone()[0]

print(f'Saved {count:,} rows → table author_stats in {DB_PATH}')

with sqlite3.connect(DB_PATH) as conn:
    pd.read_sql('SELECT * FROM author_stats ORDER BY paper_count DESC LIMIT 10', conn)


Saved 73,254 rows → table author_stats in data/arxiv.db
